In [1]:
import pandas as pd 
import sklearn
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 
from IPython.display import Image 
%matplotlib inline

In [2]:
URL = "https://github.com/ageron/handson-ml2/blob/master/datasets/housing/housing.csv?raw=true"
df = pd.read_csv(URL)
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [5]:
from sklearn.model_selection import train_test_split    
train_set, test_set = train_test_split(df, test_size=0.2, random_state=30)

housing = train_set.drop('median_house_value', axis=1)
housing_labels = train_set['median_house_value'].copy()
housing_number_data = housing.drop('ocean_proximity', axis=1)

In [6]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6# bu datadan shu indexlardagilarni ajratib olyapmiz

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True):
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self,X, y=None):
        return self
    def transform(self, X):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]# honalarni oilaga bulyapmiz shunda har bir oilaga nechta honaligi malum buladi
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:# add_bedrooms_per_room ustuni ixtiyoriy buladi
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]


## biz narigi kodimizda bu jarayonlarni qilgandik endi shularni yana choqirib avtomatlashtirishga harakat qilamiz

# pipleni bitta bitta datani organib tozalashlarni ozi avtomatlashtiradi

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler #yoki min_max_scalerdan ham foydalansa buladi

num_pipeline = Pipeline([
    ('median_qiymatlar', SimpleImputer(strategy='median')),# 1-qadam
    ('AttributesAdder', CombinedAttributesAdder(add_bedrooms_per_room = True)),
    ('standar_scaler_bir_deopazonga_tushurish', StandardScaler())
])

In [9]:
num_pipeline.fit_transform(housing_number_data)
num_pipeline

Pipeline(steps=[('median_qiymatlar', SimpleImputer(strategy='median')),
                ('AttributesAdder', CombinedAttributesAdder()),
                ('standar_scaler_bir_deopazonga_tushurish', StandardScaler())])

## endi sonli datalarga uzgartirish kiritdik bizda str data ham bor uniyam koramiz

# biz katta kanver yasab tepadagini ham uni ichiga solib yuborishimiz kerak

In [10]:
from sklearn.compose import ColumnTransformer

num_atributs = list(housing_number_data)
str_atributs = ['ocean_proximity']

full_pipeline = ColumnTransformer([
    ('num', num_pipeline, num_atributs),# 1-nomi 2-pipelineni choqiramiz 3-qaysi ustunlar ustida pipeline ishlaganini aytypapmiz
    ('categorical', OneHotEncoder(), str_atributs)
])

# endi qurqmay asosiy datani ham bersak buladi

In [12]:
housing_prepared = full_pipeline.fit_transform(housing)
housing_prepared[0:5, :]

array([[ 0.69045468, -0.74771271,  0.82109487, -0.92873698, -0.93634789,
        -0.66202411, -0.95892277, -0.50117396, -0.35013262,  0.17603889,
         0.37706385,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.73042911, -0.79459048,  0.50456328, -0.46504724, -0.56167866,
        -0.52340119, -0.56608836,  0.36500115,  0.11173742, -0.01141653,
        -0.42460338,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 0.80538117, -0.76646382, -0.36589859, -0.64951611, -0.85998219,
        -0.73357142, -0.82535907,  1.52370993,  0.45934877,  0.01634478,
        -1.05766842,  1.        ,  0.        ,  0.        ,  0.        ,
         0.        ],
       [ 1.24509991, -1.37587489, -0.68243018, -0.20413593, -0.31349013,
         0.30028727, -0.2806287 ,  0.11206675,  0.05881522,  0.11950524,
        -0.4395989 ,  0.        ,  0.        ,  0.        ,  0.        ,
         1.        ],
       [ 0.72543231, -0.67739605,  1

## endi malumotlarimizda oxiridagi 0 0 0 1 shunaqa ustunlar onehotencoder qilgan ish

## biz otgan safarga noutbookda qilgan barcha ishimizni avtomatlashtirib pipelinega tiqib quydik

# Mana endi malumotlarimiz Ml ga tayyor